# Travaux pratiques — Decision Trees, Random Forests & KNN

**Machine Learning · Classification supervisée · Notebook guidé**  
**Auteur : Savadogo Modeste**

Ce TP accompagne le cours théorique. L'objectif n'est pas seulement d'exécuter trois estimateurs de `scikit-learn`, mais de comprendre comment les utiliser, comment leurs hyperparamètres modifient leur comportement et comment comparer leurs performances sans biaiser l'évaluation.

## Objectifs d'apprentissage

À la fin du TP, vous devrez être capables de :

1. entraîner un arbre de décision et contrôler sa complexité ;
2. expliquer ce que la forêt aléatoire ajoute à un arbre individuel ;
3. calculer les voisins d'une observation et comprendre l'effet de la mise à l'échelle ;
4. construire des pipelines adaptés à chaque algorithme ;
5. sélectionner les hyperparamètres sur les données d'entraînement, puis évaluer une seule fois sur le test.

## Fil conducteur

**Petit exemple calculé → implémentation scikit-learn → observation du comportement → expérimentation sur un même dataset.**

> **Consigne de travail :** avant chaque cellule importante, lisez la question et formulez une hypothèse. Exécutez ensuite le code, observez le résultat et expliquez-le. Ne modifiez qu'un paramètre à la fois.

## 0. Environnement et conventions

Nous utilisons `numpy`, `pandas`, `matplotlib` et `scikit-learn`. Le dataset retenu est inclus dans scikit-learn : aucun téléchargement ni compte externe n'est nécessaire.

Dans tout le notebook :

- `X` désigne la matrice des caractéristiques (*features*) ;
- `y` désigne la variable cible (*target*) ;
- `X_train, y_train` servent à apprendre et régler les modèles ;
- `X_test, y_test` sont réservés à l'évaluation finale.

In [ ]:
# Si une bibliothèque manque dans votre environnement, décommentez la ligne suivante :
# %pip install -q numpy pandas matplotlib scikit-learn

import sys
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn import __version__ as sklearn_version
from sklearn.datasets import load_wine, make_moons
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Python :", sys.version.split()[0])
print("scikit-learn :", sklearn_version)

# 1. Dataset de travail — Wine recognition

Nous utiliserons le **Wine dataset** de scikit-learn. Il s'agit d'un problème de classification à trois classes : prédire la classe d'un échantillon à partir de mesures chimiques.

| Propriété | Valeur |
|---|---:|
| Nombre d'observations | 178 |
| Nombre de caractéristiques | 13 |
| Nombre de classes | 3 |
| Types de variables | Numériques continues |

Ce choix est volontaire : les trois modèles peuvent être appliqués aux mêmes variables sans consacrer la séance à l'encodage de variables catégorielles. Les échelles des caractéristiques sont toutefois très différentes, ce qui permet de voir concrètement pourquoi KNN doit généralement être précédé d'une normalisation.

**Limite à garder en tête :** 178 observations constituent un petit benchmark pédagogique. Les scores obtenus servent à comparer des comportements dans cet exercice, pas à établir une conclusion générale sur la supériorité d'un algorithme.

In [ ]:
wine = load_wine(as_frame=True)
X = wine.data.copy()
y = wine.target.copy()
class_names = list(wine.target_names)

print("Dimensions de X :", X.shape)
print("Nombre de classes :", y.nunique())
print("Noms des classes :", class_names)
print("\nRépartition des classes :")
display(y.value_counts().sort_index().rename(index=dict(enumerate(class_names))).to_frame("effectif"))
print("Aperçu des caractéristiques :")
display(X.head())

## 1.1. Observer les données avant de modéliser

**Question.** Les variables numériques sont-elles exprimées sur des échelles comparables ? Quelles caractéristiques semblent avoir les amplitudes les plus grandes ?

Une distance euclidienne additionne les écarts sur toutes les dimensions. Si une variable a une amplitude beaucoup plus grande que les autres, elle peut dominer la distance calculée.

In [ ]:
print("Statistiques descriptives :")
display(X.describe().T.round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
y.value_counts().sort_index().plot(kind="bar", ax=axes[0])
axes[0].set_title("Répartition des trois classes")
axes[0].set_xlabel("Classe")
axes[0].set_ylabel("Nombre d'observations")
axes[0].set_xticklabels(class_names, rotation=0)
axes[0].grid(axis="y", alpha=.25)

X.std().sort_values().plot(kind="barh", ax=axes[1])
axes[1].set_title("Écart-type des caractéristiques")
axes[1].set_xlabel("Écart-type")
axes[1].grid(axis="x", alpha=.25)
plt.tight_layout()
plt.show()

## 1.2. Séparation entraînement / test

Nous réservons 25 % des observations au test final. `stratify=y` maintient approximativement la proportion des trois classes dans les deux ensembles.

**Règle méthodologique :** le test ne sert ni à choisir `max_depth`, ni à choisir `n_estimators`, ni à choisir `k`. Ces choix seront faits par validation croisée sur `X_train, y_train`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

print("Train :", X_train.shape, "| Test :", X_test.shape)
print("\nProportions des classes — train :")
display(y_train.value_counts(normalize=True).sort_index().rename(index=dict(enumerate(class_names))).round(3))
print("Proportions des classes — test :")
display(y_test.value_counts(normalize=True).sort_index().rename(index=dict(enumerate(class_names))).round(3))

# 2. Decision Tree — apprendre des règles de séparation

Un arbre de décision construit une suite de tests sur les caractéristiques. À chaque nœud interne, une condition oriente l'observation vers un enfant ; une feuille fournit la prédiction.

Nous commençons par reconstruire un petit arbre de **régression** par calcul, puis nous utiliserons `DecisionTreeClassifier` sur le Wine dataset. Cette distinction est importante : la structure de l'arbre reste la même, mais le critère de séparation et la prédiction d'une feuille dépendent de la tâche.

## 2.1. Reconstruction manuelle de CART — exemple de régression

Considérons les six observations ci-dessous. Il n'y a qu'une caractéristique `x`, donc le problème consiste à rechercher les meilleurs seuils de séparation.

| Instance | $x$ | $y$ |
|---:|---:|---:|
| 1 | 2.0 | 1 |
| 2 | 1.0 | 2 |
| 3 | 0.5 | 6 |
| 4 | 2.0 | 3 |
| 5 | 3.0 | 4 |
| 6 | 1.0 | 2 |

Pour un nœud contenant les cibles $y_1,\ldots,y_m$, la prédiction constante qui minimise la somme des carrés est leur moyenne. L'erreur du nœud est :

\[
SSE(D)=\sum_{i=1}^{m}(y_i-\bar y_D)^2.
\]

Pour un seuil $s$, CART forme deux groupes ($x\leq s$ et $x>s$) et calcule la somme des SSE des deux enfants. Il retient le seuil qui minimise cette quantité.

In [ ]:
D = pd.DataFrame({
    "instance": [1, 2, 3, 4, 5, 6],
    "x": [2.0, 1.0, 0.5, 2.0, 3.0, 1.0],
    "y": [1.0, 2.0, 6.0, 3.0, 4.0, 2.0],
})
display(D)


def sse(values):
    """Somme des carrés des écarts à la moyenne d'un nœud."""
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return 0.0
    return float(((values - values.mean()) ** 2).sum())


def candidate_thresholds(values):
    """Milieux entre les valeurs distinctes successives, triées."""
    unique_values = np.sort(pd.Series(values).dropna().unique().astype(float))
    return (unique_values[:-1] + unique_values[1:]) / 2


def split_sse(frame, feature, threshold, target="y"):
    left = frame.loc[frame[feature] <= threshold, target].to_numpy()
    right = frame.loc[frame[feature] > threshold, target].to_numpy()
    return sse(left) + sse(right), left, right

print(f"Moyenne à la racine : {D.y.mean():.3f}")
print(f"SSE à la racine : {sse(D.y):.3f}")

### Générer et tester les seuils candidats

Les seuils candidats sont les milieux entre deux valeurs distinctes successives de `x` :

\[
s_k=\frac{v_k+v_{k+1}}{2}.
\]

**Avant d'exécuter :** quelles sont les valeurs distinctes de `x` ? Combien de seuils l'algorithme doit-il tester à la racine ?

In [ ]:
thresholds = candidate_thresholds(D["x"])
print("Valeurs distinctes :", np.sort(D.x.unique()))
print("Seuils candidats :", thresholds)

rows = []
for s in thresholds:
    total_sse, left, right = split_sse(D, "x", s)
    rows.append({
        "seuil": s,
        "effectif gauche": len(left),
        "moyenne gauche": left.mean() if len(left) else np.nan,
        "SSE gauche": sse(left),
        "effectif droit": len(right),
        "moyenne droite": right.mean() if len(right) else np.nan,
        "SSE droite": sse(right),
        "SSE totale": total_sse,
    })

root_table = pd.DataFrame(rows)
display(root_table.round(4))
best_root = float(root_table.loc[root_table["SSE totale"].idxmin(), "seuil"])
print(f"Meilleur seuil à la racine : x <= {best_root:g}")

### Interpréter la première séparation

Le tableau montre que le seuil $0.75$ minimise la somme des erreurs quadratiques dans les deux enfants. Nous avons construit le premier test, mais pas encore l'arbre complet : il faut examiner les enfants qui peuvent encore être divisés.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.scatter(D.x, D.y, s=85)
for row in D.itertuples():
    ax.annotate(str(row.instance), (row.x, row.y), xytext=(6, 6), textcoords="offset points")
ax.axvline(best_root, linestyle="--", label=f"Seuil retenu = {best_root:g}")
ax.set(xlabel="Caractéristique x", ylabel="Cible y", title="Première partition de la racine")
ax.legend(); ax.grid(alpha=.25); plt.show()

### Recommencer dans le fils droit

Le fils gauche ne contient qu'une observation : il ne peut pas être séparé en deux groupes non vides. Le fils droit contient cinq observations. Nous y répétons exactement la même procédure.

In [ ]:
D_left = D.loc[D.x <= best_root].copy()
D_right = D.loc[D.x > best_root].copy()
print("Fils gauche :"); display(D_left)
print("Fils droit :"); display(D_right)

right_rows = []
for s in candidate_thresholds(D_right.x):
    total_sse, left, right = split_sse(D_right, "x", s)
    right_rows.append({
        "seuil": s,
        "cibles gauche": left.tolist(),
        "SSE gauche": sse(left),
        "cibles droite": right.tolist(),
        "SSE droite": sse(right),
        "SSE totale": total_sse,
    })
right_table = pd.DataFrame(right_rows)
display(right_table.round(4))
best_right = float(right_table.loc[right_table["SSE totale"].idxmin(), "seuil"])
print(f"Meilleur seuil dans le fils droit : x <= {best_right:g}")

### Arbre obtenu et fonction de prédiction

Les feuilles prédisent la moyenne des cibles qui leur sont associées. Avec une profondeur maximale de 2 (racine à la profondeur 0), l'arbre obtenu est :

\[
\hat f(x)=\begin{cases}
6,&x\leq 0.75,\\
2,&0.75<x\leq 2.5,\\
4,&x>2.5.
\end{cases}
\]

Cette fonction en escalier est une conséquence directe des partitions successives.

In [ ]:
leaf_values = [
    D.loc[D.x <= best_root, "y"].mean(),
    D.loc[(D.x > best_root) & (D.x <= best_right), "y"].mean(),
    D.loc[D.x > best_right, "y"].mean(),
]
print("Valeurs des trois feuilles :", leaf_values)

x_grid = np.linspace(0.3, 3.2, 400)
y_hat = np.select(
    [x_grid <= best_root,
     (x_grid > best_root) & (x_grid <= best_right),
     x_grid > best_right],
    leaf_values,
)
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.scatter(D.x, D.y, s=75, label="Observations")
ax.step(x_grid, y_hat, where="post", linewidth=2.5, label="Prédiction CART")
ax.axvline(best_root, linestyle=":", color="grey")
ax.axvline(best_right, linestyle=":", color="grey")
ax.set(xlabel="x", ylabel="y / prédiction", title="Fonction de prédiction de l'arbre")
ax.legend(); ax.grid(alpha=.25); plt.show()

### Vérifier avec scikit-learn

`DecisionTreeRegressor` construit un arbre de régression. Nous vérifions que les seuils et les prédictions de feuilles coïncident avec notre calcul manuel.

In [ ]:
tree_reg = DecisionTreeRegressor(max_depth=2, random_state=RANDOM_STATE)
tree_reg.fit(D[["x"]], D["y"])
internal = tree_reg.tree_.children_left != -1
print("Seuils internes appris :", tree_reg.tree_.threshold[internal])
print("Prédiction pour les six observations :", tree_reg.predict(D[["x"]]).round(2))

fig, ax = plt.subplots(figsize=(10, 4.5))
plot_tree(tree_reg, feature_names=["x"], filled=True, rounded=True, precision=2, ax=ax)
ax.set_title("Arbre CART de régression appris par scikit-learn")
plt.show()

## 2.2. Pour la classification : impureté de Gini

En classification, une feuille prédit généralement la classe majoritaire. Pour choisir une séparation, l'arbre cherche à rendre les enfants plus homogènes en termes de classes.

\[
Gini(D)=1-\sum_{c=1}^{C}p_c^2.
\]

L'impureté d'une partition est pondérée par les effectifs des enfants :

\[
Gini_{split}=\frac{n_L}{n}Gini(D_L)+\frac{n_R}{n}Gini(D_R).
\]

**Exemple.** À la racine, nous avons 3 observations de chaque classe. Comparez deux séparations candidates.

In [ ]:
def gini_from_counts(counts):
    counts = np.asarray(counts, dtype=float)
    if counts.sum() == 0:
        return 0.0
    p = counts / counts.sum()
    return float(1 - np.sum(p ** 2))


def weighted_gini(children):
    n_total = sum(sum(c) for c in children)
    return sum(sum(c) / n_total * gini_from_counts(c) for c in children)

parent = [3, 3]
splits = {
    "A": ([2, 0], [1, 3]),
    "B": ([1, 2], [2, 1]),
}
print("Gini à la racine :", round(gini_from_counts(parent), 4))
for name, children in splits.items():
    print(f"Gini pondéré de la partition {name} : {weighted_gini(children):.4f}")

## 2.3. Implémenter un arbre de classification sur Wine

Nous passons maintenant à `DecisionTreeClassifier`.

- `criterion="gini"` choisit le critère d'impureté ;
- `max_depth` limite la profondeur ;
- `min_samples_leaf` impose un effectif minimal dans chaque feuille.

Les arbres de décision n'ont généralement pas besoin de standardisation : une transformation monotone des échelles ne change pas l'ordre des valeurs et donc l'ordre des seuils candidats.

In [ ]:
tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=3,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
)
tree.fit(X_train, y_train)

print("Accuracy train :", round(tree.score(X_train, y_train), 3))
print("Le jeu de test reste volontairement non consulté à cette étape.")

### Visualiser les règles apprises

Chaque nœud interne indique notamment la caractéristique et le seuil du test, l'impureté, le nombre d'observations et la classe prédite. Nous limitons l'affichage à trois niveaux pour garder le schéma lisible.

In [ ]:
fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(
    tree,
    feature_names=list(X.columns),
    class_names=class_names,
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8,
    ax=ax,
)
ax.set_title("Decision Tree — premiers niveaux")
plt.tight_layout(); plt.show()

### Expérience — effet de la profondeur

**Question avant exécution :** que prévoyez-vous pour un arbre très peu profond ? Et pour un arbre sans limite de profondeur ?

Nous comparons le score d'entraînement et le score moyen de validation croisée. Le jeu de test reste à l'écart.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
depths = [1, 2, 3, 4, 6, None]
depth_rows = []
for depth in depths:
    estimator = DecisionTreeClassifier(
        max_depth=depth, min_samples_leaf=2, random_state=RANDOM_STATE
    )
    scores = cross_validate(
        estimator, X_train, y_train, cv=cv, scoring="f1_macro",
        return_train_score=True, n_jobs=-1,
    )
    depth_rows.append({
        "max_depth": str(depth),
        "F1 macro train": scores["train_score"].mean(),
        "F1 macro validation": scores["test_score"].mean(),
        "écart-type validation": scores["test_score"].std(),
    })
depth_results = pd.DataFrame(depth_rows)
display(depth_results.round(3))

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(depth_results["max_depth"], depth_results["F1 macro train"], marker="o", label="Train")
ax.plot(depth_results["max_depth"], depth_results["F1 macro validation"], marker="o", label="Validation (CV)")
ax.set(xlabel="max_depth", ylabel="F1 macro", title="Complexité de l'arbre et généralisation")
ax.legend(); ax.grid(alpha=.25); plt.show()

**À retenir.** Un arbre trop peu profond peut sous-apprendre ; un arbre très profond peut mémoriser les particularités du train. `max_depth` et `min_samples_leaf` contrôlent la complexité. Une valeur ne doit pas être choisie à partir du score test.

**Mini-défi :** remplacez `min_samples_leaf=2` par `1`, puis par `5`. Observez l'effet sur le score train et le score de validation.

# 3. Random Forest — combiner plusieurs arbres

Une forêt aléatoire entraîne plusieurs arbres différents et agrège leurs décisions. Deux sources de diversité sont importantes :

1. **Bootstrap :** chaque arbre est entraîné sur un échantillon tiré avec remise ;
2. **Sélection aléatoire des caractéristiques :** à chaque nœud, seule une partie des variables est considérée pour chercher une séparation.

Le but n'est pas simplement d'augmenter le nombre d'arbres. Il faut que les arbres soient suffisamment compétents et suffisamment différents.

## 3.1. Visualiser un échantillon bootstrap

Supposons que le dataset contienne les observations $\{1,2,3,4,5,6\}$. Un échantillon bootstrap de taille 6 effectue six tirages avec remise. Une observation peut donc apparaître plusieurs fois ou ne pas apparaître.

In [ ]:
ids = np.arange(1, 7)
rng = np.random.default_rng(RANDOM_STATE)
bootstrap_samples = [rng.choice(ids, size=len(ids), replace=True) for _ in range(3)]

for index, sample in enumerate(bootstrap_samples, start=1):
    selected = np.unique(sample)
    oob = np.setdiff1d(ids, selected)
    print(f"Arbre {index} : tirages = {sample.tolist()}")
    print(f"          observations distinctes = {selected.tolist()} | OOB = {oob.tolist()}")

Pour un dataset de $n$ observations, la probabilité qu'une observation donnée ne soit jamais tirée après $n$ tirages avec remise vaut :

\[
P(\text{OOB})=\left(1-\frac{1}{n}\right)^n\longrightarrow e^{-1}\approx 0.368.
\]

En moyenne, environ 36,8 % des observations distinctes sont donc hors échantillon pour un arbre donné. Elles peuvent servir à estimer son erreur OOB.

In [ ]:
n = 1000
p_oob = (1 - 1 / n) ** n
print(f"Probabilité OOB pour n={n} : {p_oob:.3%}")
print(f"Proportion attendue incluse dans le bootstrap : {1 - p_oob:.3%}")

## 3.2. Implémenter une Random Forest

Les principaux hyperparamètres à connaître :

- `n_estimators` : nombre d'arbres ;
- `max_features` : nombre de caractéristiques candidates à chaque séparation ;
- `max_depth` : profondeur maximale de chaque arbre ;
- `min_samples_leaf` : effectif minimal d'une feuille ;
- `bootstrap=True` : tirage avec remise ;
- `oob_score=True` : calcul d'une estimation OOB.

In [ ]:
forest = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",
    max_depth=None,
    min_samples_leaf=2,
    bootstrap=True,
    oob_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
forest.fit(X_train, y_train)

print("Accuracy train :", round(forest.score(X_train, y_train), 3))
print("Score OOB       :", round(forest.oob_score_, 3))

### Observer le vote des arbres

Pour une observation test, chaque arbre produit une classe. La forêt agrège ces décisions par vote majoritaire (et `predict_proba` agrège les probabilités estimées par les arbres).

**Question.** Les 200 arbres votent-ils tous pour la même classe ? Que signifie un vote partagé ?

In [ ]:
query = X_test.iloc[[0]]
individual_votes = np.array([estimator.predict(query.to_numpy())[0] for estimator in forest.estimators_])
vote_counts = pd.Series(individual_votes).value_counts().sort_index()
print("Index de l'observation test :", X_test.index[0])
print("Classe réelle :", class_names[int(y_test.iloc[0])])
print("Votes par classe :")
for class_id, count in vote_counts.items():
    print(f"  {class_names[int(class_id)]} : {count} arbres")
print("Prédiction de la forêt :", class_names[int(forest.predict(query)[0])])

## 3.3. Visualiser l'effet de l'ensemble — frontières de décision

Le Wine dataset possède 13 caractéristiques : sa frontière de décision ne peut pas être dessinée directement dans le plan. Pour visualiser la géométrie des trois modèles, nous utilisons donc un **petit jeu synthétique à deux dimensions** (`make_moons`). Ce jeu sert uniquement à la visualisation ; l'évaluation finale reste faite sur Wine.

In [ ]:
X_moon, y_moon = make_moons(n_samples=350, noise=0.25, random_state=RANDOM_STATE)
X_m_train, X_m_test, y_m_train, y_m_test = train_test_split(
    X_moon, y_moon, test_size=0.25, stratify=y_moon, random_state=RANDOM_STATE
)

models_2d = {
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=150, max_features="sqrt", min_samples_leaf=2,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "KNN (standardisé)": Pipeline([
        ("scale", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=15)),
    ]),
}
for model in models_2d.values():
    model.fit(X_m_train, y_m_train)


def plot_boundary(model, X_plot, y_plot, title, ax):
    x_min, x_max = X_plot[:, 0].min() - .5, X_plot[:, 0].max() + .5
    y_min, y_max = X_plot[:, 1].min() - .5, X_plot[:, 1].max() + .5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 220),
        np.linspace(y_min, y_max, 220),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=.22)
    ax.scatter(X_plot[:, 0], X_plot[:, 1], c=y_plot, edgecolor="k", s=18)
    ax.set_title(title)
    ax.set_xlabel("x₁"); ax.set_ylabel("x₂")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, model) in zip(axes, models_2d.items()):
    plot_boundary(model, X_m_test, y_m_test, name, ax)
plt.tight_layout(); plt.show()

for name, model in models_2d.items():
    print(f"{name:20s} | accuracy test = {model.score(X_m_test, y_m_test):.3f}")

### Importance des caractéristiques — première lecture

Une forêt expose `feature_importances_`, calculée ici par diminution moyenne d'impureté. Cette mesure décrit l'utilisation des variables dans les arbres ; elle ne prouve pas une relation causale et peut favoriser les variables ayant davantage de possibilités de séparation.

In [ ]:
importances = pd.Series(forest.feature_importances_, index=X.columns).sort_values().tail(10)
ax = importances.plot(kind="barh", figsize=(8, 4.5))
ax.set_xlabel("Importance par diminution d'impureté")
ax.set_title("10 caractéristiques les plus utilisées par la forêt")
ax.grid(axis="x", alpha=.25)
plt.show()

**Mini-défi Random Forest :** comparez `n_estimators=20`, `100` et `300`, en gardant les autres paramètres identiques. Regardez si le score OOB et les prédictions se stabilisent. Le nombre d'arbres n'est pas un paramètre qui doit être choisi sur le jeu de test.

# 4. K-Nearest Neighbors — prédire à partir des voisins

KNN (*k-Nearest Neighbors*) est un algorithme dit *lazy learning* : l'apprentissage consiste principalement à mémoriser les observations. Pour une nouvelle observation, il calcule les distances aux observations d'entraînement, retient les $k$ plus proches et agrège leurs étiquettes.

En classification, la règle usuelle est le vote majoritaire :

\[
\hat y(x)=\operatorname{mode}\{y_i : x_i\in N_k(x)\}.
\]

## 4.1. Calculer une prédiction à la main

Nous considérons cinq observations décrites par deux caractéristiques et un nouveau point $x=(3.2,2.2)$. Avec $k=3$, calculez les distances euclidiennes, classez les observations et faites le vote majoritaire.

\[
d_2(x,z)=\sqrt{\sum_{j=1}^{p}(x_j-z_j)^2}.
\]

In [ ]:
knn_toy = pd.DataFrame({
    "observation": ["A", "B", "C", "D", "E"],
    "x1": [1, 2, 2, 4, 5],
    "x2": [1, 1, 3, 3, 4],
    "classe": ["Non", "Non", "Oui", "Oui", "Oui"],
})
query_point = np.array([3.2, 2.2])
X_toy = knn_toy[["x1", "x2"]].to_numpy(dtype=float)
distances = np.sqrt(((X_toy - query_point) ** 2).sum(axis=1))
knn_toy["distance"] = distances
ranked = knn_toy.sort_values("distance").reset_index(drop=True)
display(ranked.round(4))

k = 3
neighbors = ranked.head(k)
prediction = neighbors["classe"].value_counts().idxmax()
print("Voisins retenus :", neighbors.observation.tolist())
print(f"Vote avec k={k} : {prediction}")

## 4.2. Changer de métrique change la notion de proximité

La distance de Minkowski est définie par :

\[
d_q(x,z)=\left(\sum_{j=1}^{p}|x_j-z_j|^q\right)^{1/q}.
\]

- $q=1$ : distance de Manhattan ;
- $q=2$ : distance euclidienne ;
- $q\rightarrow\infty$ : distance de Tchebychev.

La métrique est un choix de modélisation : elle peut modifier l'ordre des voisins et donc la prédiction.

In [ ]:
def minkowski_distance(a, b, q=2):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if np.isinf(q):
        return float(np.max(np.abs(a - b)))
    return float(np.sum(np.abs(a - b) ** q) ** (1 / q))

for q in [1, 2, 3, np.inf]:
    distances_q = [minkowski_distance(query_point, row, q=q) for row in X_toy]
    print(f"q={q!s:>3} :", np.round(distances_q, 3))

## 4.3. Pourquoi mettre les caractéristiques à l'échelle ?

Les caractéristiques du Wine dataset ont des amplitudes très différentes. Une distance euclidienne calculée directement sur les données peut être dominée par une variable comme `proline`, dont les valeurs sont de plusieurs centaines ou milliers, alors que `hue` varie autour de 1.

Nous comparons le voisin le plus proche d'une observation test :

1. distances calculées sur les valeurs brutes ;
2. distances calculées après standardisation.

**Important :** le `StandardScaler` est ajusté sur `X_train` uniquement. L'observation test est transformée avec ce scaler, sans participer au calcul de la moyenne et de l'écart-type.

In [ ]:
scaler = StandardScaler().fit(X_train)
query_df = X_test.iloc[[0]]

raw_distances = np.linalg.norm(
    X_train.to_numpy() - query_df.to_numpy(), axis=1
)
scaled_train = scaler.transform(X_train)
scaled_query = scaler.transform(query_df)
scaled_distances = np.linalg.norm(
    scaled_train - scaled_query, axis=1
)

raw_order = np.argsort(raw_distances)[:5]
scaled_order = np.argsort(scaled_distances)[:5]

comparison = pd.DataFrame({
    "voisin brut": X_train.index[raw_order],
    "distance brute": raw_distances[raw_order],
    "classe brute": [class_names[int(y_train.loc[i])] for i in X_train.index[raw_order]],
    "voisin standardisé": X_train.index[scaled_order],
    "distance standardisée": scaled_distances[scaled_order],
    "classe standardisée": [class_names[int(y_train.loc[i])] for i in X_train.index[scaled_order]],
})
print("Observation test interrogée :", X_test.index[0])
display(comparison.round(3))

## 4.4. Implémenter KNN avec un Pipeline

La standardisation doit être effectuée dans un `Pipeline`. Ainsi, lors de la validation croisée, le scaler est réajusté uniquement sur la partie entraînement de chaque fold. Cela évite une fuite d'information.

Nous comparons volontairement :

- KNN sans mise à l'échelle ;
- KNN avec `StandardScaler`.

In [ ]:
knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_scaled = Pipeline([
    ("scale", StandardScaler()),
    ("model", KNeighborsClassifier(n_neighbors=5)),
])

# Comparaison par validation croisée sur le train : aucun score test n'est utilisé ici.
for name, estimator in [("Sans mise à l'échelle", knn_raw), ("Avec StandardScaler", knn_scaled)]:
    scores = cross_validate(
        estimator, X_train, y_train, cv=cv, scoring="f1_macro",
        return_train_score=False, n_jobs=-1,
    )
    print(f"{name:24s} | F1 macro CV = {scores['test_score'].mean():.3f} \u00b1 {scores['test_score'].std():.3f}")

### Expérience — influence du nombre de voisins $k$

- $k$ très petit : décision locale, potentiellement sensible au bruit ;
- $k$ plus grand : décision plus lissée, mais risque de négliger des structures locales.

Nous testons plusieurs valeurs de $k$ par validation croisée, sur les seules données d'entraînement. La standardisation est incluse dans le pipeline.

In [ ]:
k_values = [1, 3, 5, 7, 11, 15, 21]
k_rows = []
for k in k_values:
    estimator = Pipeline([
        ("scale", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k)),
    ])
    scores = cross_validate(
        estimator, X_train, y_train, cv=cv,
        scoring="f1_macro", return_train_score=True, n_jobs=-1,
    )
    k_rows.append({
        "k": k,
        "F1 macro train": scores["train_score"].mean(),
        "F1 macro validation": scores["test_score"].mean(),
        "écart-type validation": scores["test_score"].std(),
    })
k_results = pd.DataFrame(k_rows)
display(k_results.round(3))

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(k_results["k"], k_results["F1 macro train"], marker="o", label="Train")
ax.plot(k_results["k"], k_results["F1 macro validation"], marker="o", label="Validation (CV)")
ax.set(xlabel="Nombre de voisins k", ylabel="F1 macro", title="Influence de k sur KNN")
ax.legend(); ax.grid(alpha=.25); plt.show()

**Mini-défi KNN :** gardez la meilleure valeur de $k$ observée en validation croisée, puis comparez `weights="uniform"` et `weights="distance"`. Expliquez pourquoi le poids attribué à un voisin change sa contribution au vote.

# 5. Comparaison rigoureuse des trois modèles sur le même problème

Nous allons maintenant construire un protocole commun sur Wine :

1. même découpage entraînement/test pour les trois algorithmes ;
2. prétraitements propres à chaque modèle ;
3. recherche des hyperparamètres par validation croisée sur le train ;
4. évaluation finale sur le test, après sélection.

Nous utiliserons le **F1-score macro** pour donner le même poids aux trois classes, quelle que soit leur fréquence.

## 5.1. Construire les pipelines

Les arbres et les forêts n'ont généralement pas besoin de normalisation. KNN en a besoin car son mécanisme dépend des distances.

In [ ]:
from sklearn.base import clone

pipelines = {
    "Decision Tree": Pipeline([
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1)),
    ]),
    "KNN": Pipeline([
        ("scale", StandardScaler()),
        ("model", KNeighborsClassifier()),
    ]),
}

param_grids = {
    "Decision Tree": {
        "model__criterion": ["gini", "entropy"],
        "model__max_depth": [2, 3, 5, None],
        "model__min_samples_leaf": [1, 2, 4],
    },
    "Random Forest": {
        "model__n_estimators": [100, 200],
        "model__max_features": ["sqrt", None],
        "model__max_depth": [None, 5],
        "model__min_samples_leaf": [1, 2],
    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 11, 15],
        "model__weights": ["uniform", "distance"],
        "model__p": [1, 2],
    },
}

print("Pipelines définis :", list(pipelines))

## 5.2. Sélectionner les hyperparamètres par GridSearchCV

`GridSearchCV` évalue chaque combinaison du paramètre sur les folds de validation, calcule le score moyen, puis réentraîne le meilleur estimateur sur tout le train.

Pour ce dataset de petite taille, les grilles sont volontairement compactes. Sur un dataset plus grand, il faudrait réduire l'espace de recherche ou utiliser `RandomizedSearchCV`.

In [ ]:
searches = {}
search_rows = []
for name, pipeline in pipelines.items():
    print(f"Recherche en cours : {name} ...")
    search = GridSearchCV(
        estimator=clone(pipeline),
        param_grid=param_grids[name],
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        refit=True,
        return_train_score=True,
    )
    search.fit(X_train, y_train)
    searches[name] = search
    search_rows.append({
        "Modèle": name,
        "Meilleurs paramètres": search.best_params_,
        "F1 macro moyen CV": search.best_score_,
        "Écart-type CV": search.cv_results_["std_test_score"][search.best_index_],
    })

display(pd.DataFrame(search_rows).set_index("Modèle").round(3))

## 5.3. Évaluation finale sur le jeu de test

Le test n'a pas participé à la recherche d'hyperparamètres. Nous évaluons maintenant chaque meilleur estimateur sur ce jeu réservé.

**Attention à l'interprétation :** avec seulement 178 observations, quelques erreurs peuvent modifier sensiblement les scores. Il faut regarder les matrices de confusion et les scores par classe, pas uniquement un classement global.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
final_models = {"Baseline": baseline}
for name, search in searches.items():
    final_models[name] = search.best_estimator_

final_rows = []
final_predictions = {}
for name, model in final_models.items():
    pred = model.predict(X_test)
    final_predictions[name] = pred
    final_rows.append({
        "Modèle": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Précision macro": precision_score(y_test, pred, average="macro", zero_division=0),
        "Rappel macro": recall_score(y_test, pred, average="macro", zero_division=0),
        "F1 macro": f1_score(y_test, pred, average="macro", zero_division=0),
    })

final_results = pd.DataFrame(final_rows).set_index("Modèle").sort_values("F1 macro", ascending=False)
display(final_results.round(3))

## 5.4. Lire les matrices de confusion

Pour une classification à trois classes, une matrice de confusion indique combien d'observations de chaque classe réelle ont été prédites dans chacune des classes.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, pred) in zip(axes, final_predictions.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred,
        display_labels=class_names,
        cmap="Blues", colorbar=False, ax=ax,
    )
    ax.set_title(name)
plt.tight_layout(); plt.show()

for name, pred in final_predictions.items():
    print(f"\nRapport de classification — {name}")
    print(classification_report(
        y_test, pred, target_names=class_names, zero_division=0, digits=3
    ))

## 5.5. Examiner les erreurs d'un modèle

Prenons le modèle qui obtient le meilleur F1 macro sur le test à titre descriptif. La sélection du modèle, en revanche, doit s'appuyer sur la validation croisée, pas sur ce résultat test.

In [ ]:
best_name = final_results.index[0]
misclassified = X_test.loc[final_predictions[best_name] != y_test].copy()
misclassified["classe réelle"] = y_test.loc[misclassified.index].map(dict(enumerate(class_names)))
misclassified["classe prédite"] = pd.Series(final_predictions[best_name], index=X_test.index).loc[misclassified.index].map(dict(enumerate(class_names)))
print("Modèle affiché :", best_name)
print("Nombre d'erreurs :", len(misclassified), "/", len(X_test))
if len(misclassified):
    display(misclassified[["classe réelle", "classe prédite"] + list(X.columns[:4])].head(10))
else:
    print("Aucune erreur sur ce petit échantillon test.")

# 6. Synthèse et exercices

## Comparaison conceptuelle

| Aspect | Decision Tree | Random Forest | KNN |
|---|---|---|---|
| Mécanisme | Tests successifs sur les variables | Agrégation de plusieurs arbres | Recherche des voisins les plus proches |
| Apprentissage | Construction d'une structure explicite | Construction de plusieurs arbres | Mémorisation des observations |
| Mise à l'échelle | Généralement inutile | Généralement inutile | Généralement nécessaire |
| Interprétabilité | Bonne pour un arbre simple | Plus limitée | Décision locale, pas de règles globales |
| Paramètres clés | `max_depth`, `min_samples_leaf`, `criterion` | `n_estimators`, `max_features`, `max_depth` | `n_neighbors`, `weights`, `metric` |

## Exercices à réaliser

### Exercice A — Decision Tree

1. Remplacez `criterion="gini"` par `criterion="entropy"`. Les règles apprises sont-elles identiques ?
2. Comparez `max_depth=1`, `3` et `None`. Décrivez le compromis entre ajustement au train et généralisation.
3. Augmentez `min_samples_leaf`. Pourquoi la taille minimale d'une feuille régularise-t-elle l'arbre ?

### Exercice B — Random Forest

1. Comparez 20, 100 et 300 arbres. Le score OOB continue-t-il de changer fortement ?
2. Comparez `max_features="sqrt"` et `max_features=None`. Que devient la diversité des arbres ?
3. Comparez un arbre et une forêt : dans quel cas l'agrégation réduit-elle surtout la variance ?

### Exercice C — KNN

1. Comparez $k=1$, $k=5$ et $k=15$.
2. Comparez `weights="uniform"` et `weights="distance"`.
3. Comparez les métriques Manhattan (`p=1`) et euclidienne (`p=2`).
4. Recommencez sans standardisation. Pourquoi le résultat change-t-il ?

### Exercice D — Interprétation

Le meilleur score ne suffit pas à choisir un modèle. Donnez un exemple de situation où vous préféreriez un arbre simple à une forêt, même si la forêt obtient un score légèrement supérieur. Faites de même pour KNN.

## Extension possible — Bank Marketing

Le dataset Bank Marketing (UCI) est une bonne étude de cas complémentaire : il comporte 45 211 observations et des caractéristiques numériques et catégorielles. Il exige toutefois de traiter l'encodage, la mise à l'échelle, le déséquilibre de la cible et la disponibilité temporelle de certaines variables. Il est donc plus adapté à une séance consacrée au pipeline complet qu'à une première implémentation ciblée de ces trois algorithmes.

# 7. Références

- Scikit-learn — [Decision Trees](https://scikit-learn.org/stable/modules/tree.html)
- Scikit-learn — [Random Forests and ensemble methods](https://scikit-learn.org/stable/modules/ensemble.html#forests-of-randomized-trees)
- Scikit-learn — [Nearest Neighbors](https://scikit-learn.org/stable/modules/neighbors.html)
- Scikit-learn — [Pipelines and composite estimators](https://scikit-learn.org/stable/modules/compose.html)
- Scikit-learn — [Wine dataset (`load_wine`)](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html)
- UCI Machine Learning Repository — [Wine recognition dataset](https://archive.ics.uci.edu/dataset/109/wine)
- UCI Machine Learning Repository — [Bank Marketing dataset](https://archive.ics.uci.edu/dataset/222/bank+marketing)